# RainTomorrow Prediction — Data Preprocessing

This notebook transforms the raw dataset into a clean, model-ready form.

**Section order:**
1. Load raw data and sort by Location + Date
2. Feature engineering *(before target drop — required for correct lag values)*
   - 2a. Calendar features from Date
   - 2b. Lag features
   - 2c. TempRange
3. Remove rows with missing target
4. Drop high-missing columns
5. Separate features / target — encode target
6. Train/test split *(before imputation — no leakage)*
7. KNN Imputer — distribution comparison
8. Location-based imputation
9. Location clustering
10. Define preprocessing pipelines
11. Save to `data/processed/`

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.cluster import KMeans

## 2. Load Raw Data

The dataset is sorted by `Location` and `Date` immediately after loading. This ordering is required so that lag features reference the correct preceding day for each location.

In [ ]:
df = pd.read_csv("../data/weatherAUS.csv")
df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values(["Location", "Date"]).reset_index(drop=True)

print("Raw shape:", df.shape)
print("Date range:", df["Date"].min(), "→", df["Date"].max())
df.head()

## 3. Feature Engineering

Feature engineering is done **before** dropping rows with a missing target. Lag features depend on consecutive days: if we dropped a row first, the following row's lag would silently skip a day and reference incorrect data.

### 3a. Calendar Features from Date

Instead of one-hot encoding thousands of unique date strings, we extract meaningful calendar components.

| Feature | Rationale |
|---|---|
| `Year` | Long-term trend (climate drift) |
| `Month` | Seasonal cycle; raw integer works well for tree splits |
| `Month_sin`, `Month_cos` | Cyclical encoding so linear models see Jan and Dec as adjacent |
| `DayOfMonth` | Intra-month variation |
| `DayOfWeek` | Minor; kept for completeness |
| `Season` | Explicit Southern-Hemisphere season label (Summer = Dec–Feb) |

The original `Date` column is dropped once these features have been extracted.

In [ ]:
df["Year"]       = df["Date"].dt.year
df["Month"]      = df["Date"].dt.month
df["DayOfMonth"] = df["Date"].dt.day
df["DayOfWeek"]  = df["Date"].dt.dayofweek   # 0 = Monday

# Cyclical encoding for Month so that December and January are treated as adjacent
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)

# Southern Hemisphere seasons
season_map = {12: "Summer", 1: "Summer", 2: "Summer",
              3: "Autumn",  4: "Autumn",  5: "Autumn",
              6: "Winter",  7: "Winter",  8: "Winter",
              9: "Spring", 10: "Spring", 11: "Spring"}
df["Season"] = df["Month"].map(season_map)

df = df.drop(columns=["Date"])

new_cal_cols = ["Year", "Month", "Month_sin", "Month_cos", "DayOfMonth", "DayOfWeek", "Season"]
print("New calendar features:", new_cal_cols)
df[new_cal_cols].head()

### 3b. Lag Features

Consecutive weather observations are not independent — yesterday's rain, humidity, and temperature influence today's forecast. Lag features make this dependency explicit.

| Feature | Description |
|---|---|
| `lag1_Rainfall` | Previous day's rainfall |
| `lag1_Humidity3pm` | Previous day's afternoon humidity |
| `roll3_Rainfall` | 3-day rolling mean rainfall (up to and including yesterday) |

Grouping by `Location` ensures lags do not bleed between different weather stations. Rows with no preceding day (first observation per location) produce `NaN` — these will be handled by the location-based imputer in Section 8.

In [ ]:
lag_vars = {"Rainfall": 1, "Humidity3pm": 1}

for col, shift in lag_vars.items():
    if col in df.columns:
        df[f"lag1_{col}"] = df.groupby("Location")[col].shift(shift)

# 3-day rolling mean of Rainfall, shifted by 1 so today's value is excluded
if "Rainfall" in df.columns:
    df["roll3_Rainfall"] = (
        df.groupby("Location")["Rainfall"]
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

lag_cols = [c for c in df.columns if c.startswith("lag") or c.startswith("roll")]
print("Lag/rolling features created:", lag_cols)
print(f"NaN introduced (first day per location): {df[lag_cols].isna().sum().sum()}")

In [ ]:
# Visual sanity check: lag1_Rainfall should closely track Rainfall shifted by 1 day
example_loc = "Sydney"
sample = df[df["Location"] == example_loc][["Rainfall", "lag1_Rainfall", "roll3_Rainfall"]].head(30)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(sample["Rainfall"].values,      label="Rainfall (today)",       alpha=0.8)
ax.plot(sample["lag1_Rainfall"].values, label="lag1_Rainfall (yest.)",  alpha=0.8, linestyle="--")
ax.plot(sample["roll3_Rainfall"].values,label="roll3_Rainfall (3-day)", alpha=0.8, linestyle=":")
ax.set_title(f"Lag feature sanity check — {example_loc} (first 30 days)")
ax.set_xlabel("Day index")
ax.legend()
plt.tight_layout()
plt.show()

### 3c. Temperature Range

In [ ]:
if "MaxTemp" in df.columns and "MinTemp" in df.columns:
    df["TempRange"] = df["MaxTemp"] - df["MinTemp"]

print("Current shape after feature engineering:", df.shape)

## 4. Remove Rows with Missing Target

In [ ]:
df = df.dropna(subset=["RainTomorrow"]).copy()
print("Shape after dropping missing target:", df.shape)

## 5. Drop High-Missing Columns

Columns with more than 40% missing values are removed.

In [ ]:
missing_ratio = df.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.4].index.tolist()

print("Columns to drop:", cols_to_drop)

df = df.drop(columns=cols_to_drop)
print("Shape after dropping high-missing columns:", df.shape)

## 6. Separate Features and Target — Encode Target

In [ ]:
target_col = "RainTomorrow"

X = df.drop(columns=[target_col])
y = df[target_col]

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)
print("Classes:", label_encoder.classes_)

## 7. Train/Test Split

The split is done **before** imputation and clustering so that no test-set information leaks into the statistics computed from the training set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train = X_train.copy()
X_test  = X_test.copy()

print("Train shape:", X_train.shape)
print("Test shape: ", X_test.shape)

In [ ]:
numeric_features     = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", numeric_features)
print()
print("Categorical features:", categorical_features)

## 8. KNN Imputer — Distribution Comparison

`Pressure9am` has around 10% missing values and is a key meteorological predictor.

Three distributions are compared:
- **Original** — observed non-NaN values (baseline)
- **Location-based** — NaN filled with per-location median from the training set
- **KNN** — NaN filled by averaging the 5 nearest neighbours in feature space

> KNN on the full 113K-row training set would be very slow. A sample of 5 000 rows is used for illustration.

In [ ]:
example_col = "Pressure9am"
print(f"Missing in X_train['{example_col}']: {X_train[example_col].isna().sum()} "
      f"({X_train[example_col].isna().mean():.1%})")

In [ ]:
rng = np.random.default_rng(42)
# Use only original weather numerics (not lag features) to keep comparison clean
original_num = [c for c in numeric_features
                if not c.startswith("lag") and not c.startswith("roll")
                and c not in ("Year", "Month", "Month_sin", "Month_cos",
                               "DayOfMonth", "DayOfWeek")]
sample_idx = rng.choice(X_train.index, size=5000, replace=False)
X_sample = X_train.loc[sample_idx, original_num].copy()

print(f"Features used for KNN comparison: {original_num}")
print(f"Missing '{example_col}' in sample: {X_sample[example_col].isna().sum()}")

In [ ]:
# Location-based fill on the sample
sample_with_loc = X_train.loc[sample_idx, original_num + ["Location"]].copy()
loc_medians_sample = X_train.groupby("Location")[example_col].median()
sample_loc_filled = (
    sample_with_loc[example_col]
    .fillna(sample_with_loc["Location"].map(loc_medians_sample))
    .fillna(X_train[example_col].median())
)

In [ ]:
# KNN fill on the sample
knn_imputer = KNNImputer(n_neighbors=5)
X_sample_knn = pd.DataFrame(
    knn_imputer.fit_transform(X_sample),
    columns=original_num,
    index=X_sample.index
)
sample_knn_filled = X_sample_knn[example_col]

In [ ]:
original_values = X_train[example_col].dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, values, title, color in zip(
    axes,
    [original_values, sample_loc_filled, sample_knn_filled],
    [f"Original (non-NaN, n={len(original_values):,})",
     f"Location-based (sample n={len(sample_loc_filled):,})",
     f"KNN (sample n={len(sample_knn_filled):,})"],
    ["steelblue", "darkorange", "seagreen"]
):
    ax.hist(values, bins=40, color=color, edgecolor="white")
    ax.set_title(title)
    ax.set_xlabel(example_col)

plt.suptitle(f"Imputation comparison: {example_col}", fontsize=13)
plt.tight_layout()
plt.show()

print(f"{'Method':<30} {'Mean':>8} {'Std':>8} {'Median':>8}")
print("-" * 56)
for label, series in [("Original", original_values),
                       ("Location-based", sample_loc_filled),
                       ("KNN", sample_knn_filled)]:
    print(f"{label:<30} {series.mean():>8.2f} {series.std():>8.2f} {series.median():>8.2f}")

## 9. Location-Based Imputation

Per-location median (numeric) or mode (categorical) computed from the **training set only**, with a global fallback. This fills residual NaN in the original weather variables and in the lag features for the first observation of each location.

In [ ]:
# Numeric: per-location median → global median fallback
loc_num_medians  = X_train.groupby("Location")[numeric_features].median()
global_num_medians = X_train[numeric_features].median()

for col in numeric_features:
    loc_map = loc_num_medians[col]
    X_train[col] = X_train[col].fillna(X_train["Location"].map(loc_map)).fillna(global_num_medians[col])
    X_test[col]  = X_test[col].fillna(X_test["Location"].map(loc_map)).fillna(global_num_medians[col])

print(f"Remaining NaN in numeric columns (X_train): {X_train[numeric_features].isnull().sum().sum()}")

In [ ]:
# Categorical: per-location mode → global mode fallback
# Location and Season are excluded: Location is the grouping key; Season has no NaN.
cat_to_impute = [c for c in categorical_features if c != "Location"]
global_cat_modes = X_train[cat_to_impute].mode().iloc[0]

loc_cat_modes = (
    X_train.groupby("Location")[cat_to_impute]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

for col in cat_to_impute:
    loc_map = loc_cat_modes[col]
    X_train[col] = X_train[col].fillna(X_train["Location"].map(loc_map)).fillna(global_cat_modes[col])
    X_test[col]  = X_test[col].fillna(X_test["Location"].map(loc_map)).fillna(global_cat_modes[col])

print(f"Total NaN remaining — X_train: {X_train.isnull().sum().sum()}, "
      f"X_test: {X_test.isnull().sum().sum()}")

## 10. Location Clustering

Locations are grouped into weather regions using K-Means on their median numeric profiles, computed from training data only. The cluster label `LocationCluster` is added as a feature and test-set locations are assigned via the fitted centroids.

In [ ]:
loc_profiles = X_train.groupby("Location")[numeric_features].median()
print(f"{len(loc_profiles)} unique locations — profile shape: {loc_profiles.shape}")

In [ ]:
cluster_scaler = StandardScaler()
loc_profiles_scaled = cluster_scaler.fit_transform(loc_profiles)

In [ ]:
inertias = []
k_range  = range(2, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    inertias.append(km.fit(loc_profiles_scaled).inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(k_range), inertias, marker="o", linewidth=2)
plt.xlabel("Number of clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method — Location Clustering")
plt.xticks(list(k_range))
plt.tight_layout()
plt.show()

In [ ]:
k = 5
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
kmeans.fit(loc_profiles_scaled)

loc_profiles["Cluster"] = [f"C{c}" for c in kmeans.labels_]

for cluster in sorted(loc_profiles["Cluster"].unique()):
    locs = loc_profiles[loc_profiles["Cluster"] == cluster].index.tolist()
    print(f"  {cluster} ({len(locs)}): {', '.join(locs)}")

In [ ]:
profile_features = ["MinTemp", "MaxTemp", "TempRange", "Rainfall",
                    "Humidity9am", "Humidity3pm", "Pressure9am", "WindGustSpeed"]
profile_features = [f for f in profile_features if f in loc_profiles.columns]

cluster_summary = loc_profiles.groupby("Cluster")[profile_features].mean()

plt.figure(figsize=(10, 4))
sns.heatmap(cluster_summary.T, annot=True, fmt=".1f", cmap="coolwarm", linewidths=0.5)
plt.title("Mean weather profile per cluster")
plt.tight_layout()
plt.show()

In [ ]:
location_cluster_map = loc_profiles["Cluster"].to_dict()

X_train["LocationCluster"] = X_train["Location"].map(location_cluster_map)
X_test["LocationCluster"]  = X_test["Location"].map(location_cluster_map)

unseen_mask = X_test["LocationCluster"].isna()
if unseen_mask.any():
    for loc in X_test.loc[unseen_mask, "Location"].unique():
        loc_mask    = (X_test["Location"] == loc) & unseen_mask
        loc_profile = X_test.loc[loc_mask, numeric_features].median().values.reshape(1, -1)
        closest     = kmeans.predict(cluster_scaler.transform(loc_profile))[0]
        X_test.loc[loc_mask, "LocationCluster"] = f"C{closest}"
    print("Unseen locations assigned to nearest centroid.")

print("Cluster distribution (X_train):")
print(X_train["LocationCluster"].value_counts().sort_index())

In [ ]:
categorical_features = categorical_features + ["LocationCluster"]
print("Updated categorical features:", categorical_features)

## 11. Define Preprocessing Pipelines

Missing values have already been filled; `SimpleImputer` acts only as a safety net for unseen edge cases at inference time.

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

preprocessor

## 12. Save Processed Data

In [ ]:
os.makedirs("../data/processed", exist_ok=True)

X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv( "../data/processed/X_test.csv",  index=False)
np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/y_test.npy",  y_test)

joblib.dump(label_encoder,        "../data/processed/label_encoder.pkl")
joblib.dump(numeric_features,     "../data/processed/numeric_features.pkl")
joblib.dump(categorical_features, "../data/processed/categorical_features.pkl")
joblib.dump(kmeans,               "../data/processed/kmeans.pkl")
joblib.dump(cluster_scaler,       "../data/processed/cluster_scaler.pkl")
joblib.dump(location_cluster_map, "../data/processed/location_cluster_map.pkl")

print("All artifacts saved to data/processed/")
print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"  Numeric features ({len(numeric_features)}):     {numeric_features}")
print(f"  Categorical features ({len(categorical_features)}): {categorical_features}")